Install new libraries

In [18]:
!pip install ddgs trafilatura
#!pip install --upgrade pip


Import Dependencies



In [19]:
import os
from openai import OpenAI 
from dotenv import load_dotenv
import json
from pprint import pprint
from  IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura


load_dotenv()

OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()
MODEL="gpt-4.1-mini"

Step1: Define the tools

In [39]:
ddgs= DDGS()
results=ddgs.text("AI in Healthcare in 2030", max_results=3)
results

[{'title': 'AI advances are set to reshape healthcare by 2030, IEEE report finds | Fox Business',
  'href': 'https://www.foxbusiness.com/technology/how-ai-could-redefine-healthcare-end-decade',
  'body': 'Artificial intelligence is set to reshape healthcare through personalized medicine, gene therapy and diagnostics, according to a new IEEE Technology Megatrends 2030 Report. Published 2 weeks ago'},
 {'title': 'AI in Healthcare: 3 Revolutionary Trends by 2030 | Kindo',
  'href': 'https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030',
  'body': 'AI transforms healthcare via predictive analytics, precision medicine, and automated diagnostics. Explore 3 key shifts defining the 2030 medical landscape. Published May 24, 2024'},
 {'title': 'Here are 3 ways AI will change healthcare by 2030 | World Economic Forum',
  'href': 'https://www.weforum.org/stories/2020/01/future-of-artificial-intelligence-healthcare-delivery/',
  'body': 'January 7, 2020 - In 2030, AI-powered predictive h

In [40]:
#url="https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare"
url="https://intelligence.weforum.org/topics/a1Gb00000038pGiEAI"
downloaded=trafilatura.fetch_url(url=url)
if downloaded:
    text=trafilatura.extract(downloaded)
print(text)

You need to enable JavaScript to run this app.


FUNCTIONS

In [ ]:
#Search web function

def search_web(query:str):
    """Search the web using DuckDuckGo and return the top 3 results."""
    ddgs= DDGS()
    results=ddgs.text(query, max_results=3)
    print(f"\u2705 Got results")
    return json.dumps(results, indent=2)




In [ ]:
#Fetch URL Function

def fetch_url(url:str):
    """ Fetch the content of URL using trafilatura and return the text content."""
    downloaded=trafilatura.fetch_url(url=url)
    if downloaded:
        text=trafilatura.extract(downloaded)
        if text:
            print(f"\u2705 Got Text: {len(text)} chars")
            return text
    print(f"\u274c Failed to fetch URL:")
    return f"Could not extract text from {url}. Try a different source"

In [24]:
#search_web("AI in Healthcare in 2030")
result=fetch_url("https://www.weforum.org/stories/2020/01/future-of-artificial-intelligence-healthcare-delivery")
print(result)

  ❌ Failed to fetch URL:
Could not extract text from https://www.weforum.org/stories/2020/01/future-of-artificial-intelligence-healthcare-delivery. Try a different source


STEP2: Describe as LLM tools

In [43]:
tools=[]

In [44]:
search_web_function={
    "name": "search_web",
    "description": "Searches the web using DuckDucGo browser. returns 3 results.",
    "parameters": {
        "type":"object",
        "properties":{
            "query":{
                "type":"string",
                "description":"Searches the websites for information related to the users query"
            }
        },
        "required":["query"]
    }
    
    }

tools.append({"type":"function","function":search_web_function})


fetch_url_function={
    "name": "fetch_url",
    "description": "Fetches the content of a URL.",
    "parameters": {
        "type":"object",
        "properties":{
            "url":{
                "type":"string",
                "description":"The URL to fetch and extract text from "      }
        },
        "required":["url"]
    }
    
    }
tools.append({"type":"function","function":fetch_url_function})


In [45]:
tools

[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Searches the web using DuckDucGo browser. returns 3 results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'Searches the websites for information related to the users query'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'Fetches the content of a URL.',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'The URL to fetch and extract text from '}},
    'required': ['url']}}}]

#Step3: Tool call handler

In [62]:
def handle_tool_call(tool_calls):
    tool_results=[]
    
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args= json.loads(tool_call.function.arguments)

        print(f"\U0001f527 calling function: {function_name} with args: {args}")

        if function_name=="search_web":
        #Send the notification, i.e. call the tool
            result=search_web(args["query"])
            content=f" Search Results: {result}"
        elif function_name == "fetch_url":
            result=fetch_url(args["url"])
            content=f" Fetched URL content: {result}"
        #elif function_name="Insert_function-3":
           # content=Insert_function-3{args["message"]}"

        else:
            content = f"unknown function: {function_name}"

   # print(f"sent notification: {args['message']}")
        tool_call_result={
            "role":"tool",
            "content": content,
            "tool_call_id":tool_call.id
        }
        tool_results.append(tool_call_result)
    return tool_results

Step4: The System Prompt

In [29]:
RESEARCH_AGENT_PROMPT_selftest = """You are a research agent that can search the web and fetch content from URLs. \
You have access to the following tools:
Search Web: Searches the web using DuckDuckGo and returns the top 3 results.
Fetch URL: Fetches the content of a URL and extracts the text content.
You are to use these tools to gather information and provide a comprehensive response to the user's query.
Use the json format to call the tools.

Important: Do not make up information. If you cannot find the information or the tool. Say you cannot find the information and provide a suggestion to the user to try a different query or URL.
"""




In [71]:
#Research agent prompt

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.


You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. You MUST gather information from at least 6 distinct sources before delivering your brief. If you have fewer than 6 sources, keep searching. 

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

It is imperative that "Done:" should be at the start of the final response, so that is can easily be parsed and extracted.
You CANNOT and SHOULD NOt include "Done:" in any part of your response except at the BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

#Step5: The Agentic Loop

In [70]:
def run_research_agent(topic:str, max_iterations: int =10) -> str:
    """
    Run the research agent on a topic and return the research brief

    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent infinite loops

    Returns:
        The research brief as a string

    """
#Initialize conversation message list with System prompt + Research Task
    print(f"\n\U0001F50D Starting research agent for topic: {topic}\n")
    messages=[{"role": "system", "content": RESEARCH_AGENT_PROMPT},
              {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief: {topic}"}]

    #Loop
    iteration=0
        #1. Call the LLM and get response
    while iteration <max_iterations:
        iteration+=1
        print(f"\n\U0001F4DD Iteration {iteration}:\n")
        response=client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools

    )

        message= response.choices[0].message
        messages.append(message)

        #2. Check if LLM called tools
        if message.tool_calls:
          tool_results = handle_tool_call(message.tool_calls) #whole list of tool calls on purpose
          messages.extend(tool_results) #.... add info about tool call response to "context", i.e messages change from append to extend for multiple tool calks
    
    #3. Otherwise: No tools were called, read message content
        else:
            content=message.content or "" #no tool calls         #... invoke the LLM one more time to get its updated response

    #Check if Done, then return
    #Otherwise not yet done, append message
    
            if content.startswith("DONE:"):
                brief=content[5:]
                print(f"\n\u2705 Research brief completed")
                return brief
            else:
                print(f"  \U0001F4DD Agent is thinking:")
                pprint(content)
        
#4. If we are entering the final iteration, force a final answer
        if (iteration == max_iterations-1):
            print(f"\n\u26A0  Safety Limit Reached. Stopping Research in next iteration")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please provide your final research brief now."})

       
#Fallback return
    return "Research brief incomplete. Maximum iterations reached without a finalizing brief."





In [73]:
#Lets run it
brief= run_research_agent("AI in Healthcare in 2030")
display(Markdown(brief))


🔍 Starting research agent for topic: AI in Healthcare in 2030


📝 Iteration 1:

🔧 calling function: search_web with args: {'query': 'AI in healthcare 2030 projections'}
  ✅ Got results

📝 Iteration 2:

🔧 calling function: fetch_url with args: {'url': 'https://www.foxbusiness.com/technology/how-ai-could-redefine-healthcare-end-decade'}
  ✅ Got Text: 3275 chars
🔧 calling function: fetch_url with args: {'url': 'https://www.startus-insights.com/innovators-guide/ai-in-healthcare/'}
  ❌ Failed to fetch URL:
🔧 calling function: fetch_url with args: {'url': 'https://www.eicta.iitk.ac.in/knowledge-hub/artificial-intelligence/future-of-ai-in-healthcare-predictions-innovations-2030'}
  ✅ Got Text: 9915 chars

📝 Iteration 3:

🔧 calling function: search_web with args: {'query': 'AI healthcare trends by 2030'}
  ✅ Got results

📝 Iteration 4:

🔧 calling function: fetch_url with args: {'url': 'https://fourweekmba.com/healthcare-is-becoming-the-most-valuable-ai-vertical-and-its-not-even-close/'}
  ✅ G

 

**Research Brief: AI in Healthcare in 2030**

**Key Facts and Statistics:**
1. The global AI in healthcare market is expected to grow significantly, reaching USD 164.16 billion by 2030, with projections as high as $1.03 trillion between 2026 and 2034 at a CAGR of approximately 37% (sources: StartUs Insights, Space-O Technologies).
2. Healthcare AI is set to become the most valuable AI vertical due to its complexity and regulatory environment, with a market of around $187 billion by 2030 (source: FourWeekMBA).

**Main Themes and Arguments:**
1. **Advancements in AI Technologies:**
   - AI will play a foundational role in healthcare by 2030, impacting patient management, diagnostics, drug discovery, and preventive care. Technologies like generative AI and deep learning will further embed AI systems in clinical decision-making (source: EICTA, Fox Business).

2. **Key Innovation Areas:**
   - **Personalized Medicine:** AI will drive precision medicine, tailoring treatment plans based on individual genetics and history, offering hyper-personalized approaches particularly in oncology (source: AI in Healthcare Predictions).
   - **Multimodal Diagnostics:** Future AI systems will integrate data from imaging, genomics, and wearables to offer comprehensive diagnostic insights, moving beyond single-modality analyses (source: Space-O Technologies).

3. **Operational Improvements:**
   - AI will expedite medical imaging, improve diagnostics accuracy, enhance surgical robotics, and reduce administrative burdens, leading to more efficient healthcare provision (source: EICTA, Space-O Technologies).

4. **Healthcare System Overhaul:**
   - The adoption of AI will align with foundational changes in the healthcare system, moving from reactive to proactive and preventive models. Predictive AI will help in early disease prediction, aiding in shifting from treatment to prevention (source: Space-O Technologies).

5. **Regulatory and Ethical Considerations:**
   - Effective AI implementation in healthcare will require navigating regulatory barriers and ensuring patient data privacy. Regulatory bodies are expected to establish frameworks for structured AI lifecycle management by 2030 (source: Space-O Technologies).

6. **Economic and Structural Impact:**
   - The intersection of healthcare with AI will create defensible sub-markets, such as clinical documentation and virtual nursing, enhancing the sector's overall structural value (source: FourWeekMBA).

**Notable Data Points:**
- AI-driven predictive analysis could enable early intervention for chronic diseases like Type 2 diabetes years before clinical onset (source: Space-O Technologies).
- Autonomous AI systems will manage entire patient journeys, including post-discharge care, without human intervention at each step (source: Space-O Technologies).
- The rapid integration of agentic AI and ambient clinical intelligence is anticipated to automate documentations and enhance decision-making without manual inputs (source: Space-O Technologies).

**Source URLs for Attribution:**
1. [Fox Business](https://www.foxbusiness.com/technology/how-ai-could-redefine-healthcare-end-decade)
2. [StartUs Insights](https://www.startus-insights.com/innovators-guide/ai-in-healthcare/)
3. [EICTA](https://www.eicta.iitk.ac.in/knowledge-hub/artificial-intelligence/future-of-ai-in-healthcare-predictions-innovations-2030)
4. [FourWeekMBA](https://fourweekmba.com/healthcare-is-becoming-the-most-valuable-ai-vertical-and-its-not-even-close/)
5. [Space-O Technologies](https://www.spaceo.ai/healthcare/future-of-ai/) 

This comprehensive analysis provides a forward-looking perspective on how AI is poised to revolutionize healthcare by 2030, through advancements in technology, operational efficiency, and systemic transformations.